In [ ]:
import datetime 
import pandas as pd
import numpy as np
import seaborn as sns

## DATETIME Class

In [ ]:

# strptime: string PARSE time (string → datetime)
# Converts a string TO a datetime object
date_string = "2026-01-25"
date_obj = datetime.datetime.strptime(date_string, '%Y-%m-%d')
print(date_obj)  # 2026-01-25 00:00:00
print(type(date_obj))  # <class 'datetime.datetime'>

# strftime: string FORMAT time (datetime → string)
# Converts a datetime object TO a formatted string
date_obj = datetime.datetime(year=2026, month=1, day=25)
formatted_string = date_obj.strftime('%Y-%m-%d')
print(formatted_string)  # "2026-01-25"
print(type(formatted_string))  # <class 'str'>

formatted_string = date_obj.strftime('%Y %B %d, %a')
print(formatted_string)  # "2026-01-25"

https://docs.python.org/3/library/datetime.html#format-codes

In [ ]:
date_string = "Jan 25, 2026"
date_obj = datetime.datetime.strptime(date_string, '%b %d, %Y')
date_obj

In [ ]:
date_string = "01-02-2026"
date_obj = datetime.datetime.strptime(date_string, '%d-%m-%Y')
date_obj.strftime('%B')

### Zadanie 1:
Jakiego dnia tygodnia się Państwo urodzili? (Pon, wt, sr... ? )

## Pandas Datetime

https://pandas.pydata.org/docs/user_guide/timeseries.html

https://pandas.pydata.org/docs/user_guide/timedeltas.html

In [ ]:
dti = pd.to_datetime(
    ["1/1/2018", np.datetime64("2018-01-01"), datetime.datetime(2018, 1, 1), np.nan]
)

In [ ]:
dti

In [ ]:
dti[0].strftime('%Y-%m-%d')

In [ ]:
dti.day_name()

## date_range
https://pandas.pydata.org/docs/reference/api/pandas.date_range.html#pandas.date_range

https://pandas.pydata.org/docs/user_guide/timeseries.html#timeseries-offset-aliases

In [ ]:
dti = pd.date_range("2026-01-25", periods=3, freq="d")
dti

In [ ]:
dti = pd.date_range("2018-01-01", periods=3, freq="YE")
dti

### Zadanie 2:

Stworz Data Frame z pierwszym i ostatnim dniem miesiace dla wszystkich miesiecy roku 2024.

* Dodaj kolumne z nazwa miesiace (January, Fabuary, etc...)
* Dodaj kolumne z dniem tygodnia dla pierwszego dnia (Monday, Tuesday, ...)
* Policz ile jest dni w miesiacu



# Online Retail

In [ ]:
df = pd.read_excel(r"./zjazd3/Online_retail/Online Retail.xlsx")

In [ ]:
df.dtypes

In [ ]:
df['InvoiceDay_name'] = df.InvoiceDate.dt.day_name()

In [ ]:
df['InvoiceMonth'] = df.InvoiceDate.dt.to_period('M')
df['InvoiceMonth_str'] =df.InvoiceDate.dt.strftime('%Y-%m')

In [ ]:
df['CustomerID'] = df['CustomerID'].astype('Int64')
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)
df['Country'] = df['Country'].astype(str)
df['Description'] = df['Description'].astype(str)

df["TotalPrice"] = df["UnitPrice"] * df['Quantity'] 

def set_invoice_type(x):
    if pd.isna(x.CustomerID) and x.InvoiceNo[0] == 'A':
        return 'Adjust bad debt'
    elif pd.isna(x.CustomerID):
        return 'lost'
    elif x.InvoiceNo[0] == 'C':
        return 'return'
    elif x.InvoiceNo[0] == '5':
        return 'order'
    else:
        return 'other'
    
df['invoice_type'] = df.apply(set_invoice_type, axis =1 )

df_invoice = df.query('invoice_type=="order"').groupby(['InvoiceNo', 'CustomerID', 'Country'], dropna=False).agg(
    InvoiceDate_start = ('InvoiceDate', "min"),
    InvoiceDate_end = ('InvoiceDate', "max"),
    total_quantity = ("Quantity", "sum"),
    unique_items = ('StockCode', 'nunique'),
    n_positions = ('InvoiceNo', 'count'),
    total_value = ('TotalPrice', 'sum')
).reset_index()

df_invoice['InvoiceMonth'] = df_invoice.InvoiceDate_start.dt.to_period('M')
df_invoice['InvoiceMonth_str'] =df_invoice.InvoiceDate_start.dt.strftime('%Y-%m')

df_invoice['InvoiceDay'] = df_invoice.InvoiceDate_start.dt.to_period('D')
df_invoice['InvoiceMDay_str'] =df_invoice.InvoiceDate_start.dt.strftime('%d')
df_invoice['InvoiceDay_of_month'] = df_invoice['InvoiceDate_start'].dt.day


In [ ]:
df_invoice

In [ ]:
import matplotlib.pyplot as plt
sns.boxplot(data=df_invoice, x='InvoiceMonth', y ='total_value', log_scale=[False, True])

plt.xticks(rotation=90)
plt.show()

In [ ]:
# Using FacetGrid
g = sns.FacetGrid(df_invoice, col='InvoiceMonth_str', col_wrap=3, height=4)
g.map(sns.boxplot, 'InvoiceDay_of_month', 'total_value')
g.set(ylim=(0, df_invoice['total_value'].quantile(0.99)))
g.set_titles('Month: {col_name}')
g.set_axis_labels('Day of Month', 'Total Value')

# Show every second label
for ax in g.axes.flat:
    labels = [label.get_text() for label in ax.get_xticklabels()]
    for i, label in enumerate(ax.get_xticklabels()):
        if i % 2 != 0:
            label.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
for month in df_invoice.InvoiceMonth_str.unique()[0:5]:
    g = sns.boxplot(data = df_invoice.query(f"InvoiceMonth_str=={month!r}"), x='InvoiceDay_of_month', y='total_value')
    g.set(ylim=(0, df_invoice['total_value'].quantile(0.99)))
    g.set_title(f"{month}")
    plt.show()

In [ ]:
for month in df_invoice.InvoiceMonth_str.unique()[0:4]:
    g = sns.boxplot(data = df_invoice.query(f"InvoiceMonth_str=={month!r}"), 
                    x='InvoiceDay_of_month', 
                    y='total_value', 
                    order=range(1, 32))
    # Show every second label

    # labels = [label.get_text() for label in g.axes.get_xticklabels()]
    # for i, label in enumerate(g.axes.get_xticklabels()):
    #     if i % 2 != 0:
    #         label.set_visible(False)
    # g.set(ylim=(0, df_invoice['total_value'].quantile(0.99)))
    g.set_title(f"{month}")
    plt.show()

### Zadanie 3: 

Stworzyc wykres z rozkladem wartosci faktur (np. boxplot jak wyzej) dla pierwszego kwartalu 2011 roku. 

Note: Caly kwartal ma byc na jednym wykresie (w przeciwienstwie do wykresow wyzej, gdzie kazdy miesiac jest osobno)

*Punkt dodatkowy:*

Na wykresie wyzej w order podane sa wartosci od 1 do 31 - dla kazdego miesiaca takie same. Jak to zmienic, tak, aby liczba dni pokazywana na wykresie odpowiadala rzeczywistej liczbie dni w danym miesiacu? 
